# MoNuSAC · Notebook 2 — reliability experiments (mirror of `hypothesis_build_up.ipynb`)

Consumes the cache from Notebook 1 and reproduces the "Confidently Wrong" pipeline on MoNuSAC.

**Imbalance policy** (Macrophage~587, Neutrophil~631 vs Epithelial/Lymphocyte~15k): NO augmentation,
NO resampling. `class_weight="balanced"` + `balanced_accuracy` + tied (pooled) precision. The organ arm
optionally collapses `{Macrophage, Neutrophil} -> Other`; the open-set arm uses the rare classes as novelty.

Cells that re-embed jittered crops (appearance arm + gate) reload CONCH — same as your file-2 stain cell.


In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

In [2]:
import numpy as np, glob
if not hasattr(np, "trapz"): np.trapz = np.trapezoid          # numpy 2.0 shim

NPZ = glob.glob("/kaggle/input/notebooks/mdnomanbiswassibly/monusac-01-conch-embeddings/monusac_conch.npz", recursive=True)
NPZ = NPZ[0] if NPZ else "/kaggle/working/monusac_conch.npz"
d = np.load(NPZ, allow_pickle=True)
emb_probe = d["emb_probe"].astype(np.float32)
labels    = d["labels"].astype(int)
organs    = np.asarray(d["organs"])
img_idx   = d["img_idx"].astype(int)
CLASS_NAMES = ["Epithelial", "Lymphocyte", "Macrophage", "Neutrophil"]; N_CLS = 4
ORGANS = ["Breast", "Kidney", "Lung", "Prostate"]
print("N =", len(labels), "| class counts:", dict(zip(CLASS_NAMES, np.bincount(labels, minlength=4))))

def grouped_split(img_idx, fracs=(0.60,0.15,0.25), seed=0):
    rng = np.random.RandomState(seed); imgs = np.unique(img_idx); rng.shuffle(imgs)
    n=len(imgs); a=int(fracs[0]*n); b=int((fracs[0]+fracs[1])*n)
    s_tr,s_va,s_te = set(imgs[:a]), set(imgs[a:b]), set(imgs[b:])
    inset=lambda s: np.fromiter((i in s for i in img_idx), bool, len(img_idx))
    return inset(s_tr), inset(s_va), inset(s_te)
tr, va, te = grouped_split(img_idx)
print("train/val/test:", tr.sum(), va.sum(), te.sum())

N = 46686 | class counts: {'Epithelial': np.int64(21721), 'Lymphocyte': np.int64(23281), 'Macrophage': np.int64(893), 'Neutrophil': np.int64(791)}
train/val/test: 27416 7822 11448


In [3]:
# ---- shared primitives (identical to your PanNuke pipeline) ----
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import balanced_accuracy_score, classification_report, roc_auc_score

def softmax(z): z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def ece(probs,y,n_bins=15):
    conf=probs.max(1); pred=probs.argmax(1); correct=(pred==y).astype(float)
    bins=np.linspace(0,1,n_bins+1); e=0.0
    for i in range(n_bins):
        m=(conf>bins[i])&(conf<=bins[i+1])
        if m.sum(): e+=abs(correct[m].mean()-conf[m].mean())*m.sum()
    return e/len(y)
def fit_temperature(logits,y,iters=100):
    T=torch.nn.Parameter(torch.ones(1)); L=torch.tensor(logits).float(); Y=torch.tensor(y).long()
    opt=torch.optim.LBFGS([T],lr=0.01,max_iter=iters)
    def closure(): opt.zero_grad(); l=torch.nn.functional.cross_entropy(L/T,Y); l.backward(); return l
    opt.step(closure); return float(T.item())
def risk_coverage(conf,correct):
    o=np.argsort(-conf); c=correct[o].astype(float); cov=np.arange(1,len(c)+1)/len(c)
    risk=1-np.cumsum(c)/np.arange(1,len(c)+1); return cov,risk,float(np.trapz(risk,cov))
def bootstrap_ci(fn,y,p,n=1000,seed=0):
    rng=np.random.RandomState(seed); idx=np.arange(len(y)); v=[fn(y[b],p[b]) for b in (rng.choice(idx,len(idx),True) for _ in range(n))]
    lo,hi=np.percentile(v,[2.5,97.5]); return float(np.mean(v)),float(lo),float(hi)
def fit_geometry(Xtr,ytr,classes):
    mus=np.stack([Xtr[ytr==c].mean(0) for c in classes]).astype(np.float32)
    resid=(Xtr-mus[[list(classes).index(c) for c in ytr]]).astype(np.float32)
    P=LedoitWolf().fit(resid).precision_.astype(np.float32); return mus,P
def maha_min(X,mus,P):
    return np.stack([np.einsum("nd,dk,nk->n",X-mus[i],P,X-mus[i]) for i in range(len(mus))],1).min(1)

In [4]:
# ---- baseline probe + calibration + risk-coverage ----
scaler=StandardScaler().fit(emb_probe[tr])
Xtr,Xva,Xte=(scaler.transform(emb_probe[m]) for m in (tr,va,te))
ytr,yva,yte=labels[tr],labels[va],labels[te]
clf=LogisticRegression(max_iter=2000,class_weight="balanced").fit(Xtr,ytr)
pred=clf.predict(Xte)
print("Balanced acc (test):",round(balanced_accuracy_score(yte,pred),3))
print(classification_report(yte,pred,target_names=CLASS_NAMES,digits=3))
m,lo,hi=bootstrap_ci(balanced_accuracy_score,yte,pred); print(f"95% CI: {m:.3f} [{lo:.3f},{hi:.3f}]")

T=fit_temperature(clf.decision_function(Xva),yva)
pa=softmax(clf.decision_function(Xte)/T); ok=pa.argmax(1)==yte
_,_,aurc=risk_coverage(pa.max(1),ok)
print(f"T={T:.3f} | ECE={ece(pa,yte):.4f} | AURC={aurc:.4f}")
for q in (0.7,0.8,0.9,1.0):
    k=int(q*len(ok)); print(f"  acc @ {int(q*100)}% cov:",round(ok[np.argsort(-pa.max(1))][:k].mean(),3))

Balanced acc (test): 0.807
              precision    recall  f1-score   support

  Epithelial      0.951     0.891     0.920      5076
  Lymphocyte      0.931     0.928     0.930      5894
  Macrophage      0.624     0.689     0.655       289
  Neutrophil      0.274     0.720     0.397       189

    accuracy                          0.902     11448
   macro avg      0.695     0.807     0.725     11448
weighted avg      0.921     0.902     0.910     11448

95% CI: 0.807 [0.787,0.828]
T=1.203 | ECE=0.0260 | AURC=0.0225
  acc @ 70% cov: 0.979
  acc @ 80% cov: 0.966
  acc @ 90% cov: 0.942
  acc @ 100% cov: 0.902


In [5]:
# ---- geometry: tied-precision Maha (epistemic) + kNN overlap (aleatoric) ----
K=20
mus,P=fit_geometry(Xtr,ytr,list(range(N_CLS)))
nn=NearestNeighbors(n_neighbors=K).fit(Xtr)
def knn_overlap(X,pred):
    _,idx=nn.kneighbors(X,n_neighbors=K); return 1.0-(ytr[idx]==pred[:,None]).mean(1)
epi_te=maha_min(Xte,mus,P); ale_te=knn_overlap(Xte,pa.argmax(1))
ok=pa.argmax(1)==yte
print("epistemic (Maha)   correct vs wrong:",round(epi_te[ok].mean(),3),"vs",round(epi_te[~ok].mean(),3))
print("aleatoric (overlap) correct vs wrong:",round(ale_te[ok].mean(),3),"vs",round(ale_te[~ok].mean(),3))

epistemic (Maha)   correct vs wrong: 339.014 vs 495.41
aleatoric (overlap) correct vs wrong: 0.146 vs 0.636


In [6]:
# ---- per-class geometry (TEST + VAL replication). Rare classes -> wide CIs, report honestly. ----
def per_class_geometry(mask,name):
    X=scaler.transform(emb_probe[mask]); y=labels[mask]; pr=clf.predict(X)
    epi=maha_min(X,mus,P); ale=knn_overlap(X,pr)
    print(f"\n[{name}] class : aleatoric(overlap)  epistemic(maha)  (mean ± boot95)")
    for c in range(N_CLS):
        m=y==c
        if m.sum()<5: print(f"  {CLASS_NAMES[c]:11s}: n={m.sum()} (too few)"); continue
        a=bootstrap_ci(lambda yy,pp:pp.mean(),y[m],ale[m]); e=bootstrap_ci(lambda yy,pp:pp.mean(),y[m],epi[m])
        print(f"  {CLASS_NAMES[c]:11s}: {a[0]:.3f} [{a[1]:.3f},{a[2]:.3f}]   {e[0]:.1f} [{e[1]:.1f},{e[2]:.1f}]  (n={m.sum()})")
per_class_geometry(te,"TEST"); per_class_geometry(va,"VAL")


[TEST] class : aleatoric(overlap)  epistemic(maha)  (mean ± boot95)
  Epithelial : 0.232 [0.224,0.240]   371.2 [364.8,377.6]  (n=5076)
  Lymphocyte : 0.117 [0.111,0.123]   322.3 [319.2,325.6]  (n=5894)
  Macrophage : 0.796 [0.765,0.824]   624.4 [598.0,650.6]  (n=289)
  Neutrophil : 0.654 [0.607,0.702]   490.4 [461.8,520.4]  (n=189)

[VAL] class : aleatoric(overlap)  epistemic(maha)  (mean ± boot95)
  Epithelial : 0.179 [0.171,0.187]   358.3 [354.6,361.7]  (n=3816)
  Lymphocyte : 0.129 [0.121,0.136]   313.3 [308.0,318.9]  (n=3754)
  Macrophage : 0.759 [0.706,0.812]   452.5 [424.8,478.9]  (n=136)
  Neutrophil : 0.678 [0.628,0.728]   660.6 [618.3,709.7]  (n=116)


In [7]:
# ---- OPEN-SET: leave-one-class-out AUROC. Rare classes are GOOD novelty sources here. ----
def loco_openset(held):
    keep=labels!=held; trk=tr&keep
    sc=StandardScaler().fit(emb_probe[trk]); Xk=sc.transform(emb_probe[trk]); yk=labels[trk]
    cls=LogisticRegression(max_iter=2000,class_weight="balanced").fit(Xk,yk)
    classes=np.unique(yk); mus_k,P_k=fit_geometry(Xk,yk,list(classes))
    Xt=sc.transform(emb_probe[te]); unk=(labels[te]==held).astype(int)
    if unk.sum()==0: return None
    lg=cls.decision_function(Xt)
    msp=1-softmax(lg).max(1); energy=-torch.logsumexp(torch.tensor(lg).float(),1).numpy(); maha=maha_min(Xt,mus_k,P_k)
    return {s:round(roc_auc_score(unk,v),3) for s,v in [("MSP",msp),("energy",energy),("maha",maha)]}, int(unk.sum())
print("OPEN-SET (lead with COMPLEMENTARITY, never 'geometry wins'):")
for h in range(N_CLS):
    r=loco_openset(h)
    if r: print(f"  novel={CLASS_NAMES[h]:11s} (n={r[1]:5d}) {r[0]}")

OPEN-SET (lead with COMPLEMENTARITY, never 'geometry wins'):
  novel=Epithelial  (n= 5076) {'MSP': np.float64(0.803), 'energy': np.float64(0.845), 'maha': np.float64(0.755)}
  novel=Lymphocyte  (n= 5894) {'MSP': np.float64(0.765), 'energy': np.float64(0.809), 'maha': np.float64(0.741)}
  novel=Macrophage  (n=  289) {'MSP': np.float64(0.574), 'energy': np.float64(0.602), 'maha': np.float64(0.891)}
  novel=Neutrophil  (n=  189) {'MSP': np.float64(0.64), 'energy': np.float64(0.721), 'maha': np.float64(0.806)}


In [8]:
# ---- ARM B: ORGAN shift, leave-one-organ-out (4 organs -> 4 folds, report per-fold) ----
COLLAPSE_RARE = True   # {Mac,Neu} -> 'Other' so held-out-organ balanced acc stays meaningful
def collapsed(labs):
    if not COLLAPSE_RARE: return labs, CLASS_NAMES
    m=labs.copy(); m[(labs==2)|(labs==3)]=2; return m, ["Epithelial","Lymphocyte","Other"]
lab, names = collapsed(labels); classes=list(range(len(names)))
print(f"ARM B (organ) over {names}\nheld-out | bal.acc | ECE  | AURC | acc@70 | Maha | held-out counts")
rows=[]
for held in ORGANS:
    ev=organs==held
    if ev.sum()==0: print(f"  {held}: absent"); continue
    idx=np.where(~ev)[0]; rng=np.random.RandomState(0); rng.shuffle(idx); cut=int(0.85*len(idx))
    trm=np.zeros(len(lab),bool); vam=np.zeros(len(lab),bool); trm[idx[:cut]]=True; vam[idx[cut:]]=True
    sc=StandardScaler().fit(emb_probe[trm])
    Xt,Xv,Xe=sc.transform(emb_probe[trm]),sc.transform(emb_probe[vam]),sc.transform(emb_probe[ev])
    cl=LogisticRegression(max_iter=2000,class_weight="balanced").fit(Xt,lab[trm])
    Tt=fit_temperature(cl.decision_function(Xv),lab[vam]); pae=softmax(cl.decision_function(Xe)/Tt)
    oke=pae.argmax(1)==lab[ev]; _,_,au=risk_coverage(pae.max(1),oke)
    acc70=oke[np.argsort(-pae.max(1))][:int(0.7*ev.sum())].mean()
    present=[c for c in classes if (lab[trm]==c).sum()>0]; mu,Pp=fit_geometry(Xt,lab[trm],present)
    mh=float(maha_min(Xe,mu,Pp).mean()); cnt=dict(zip(names,np.bincount(lab[ev],minlength=len(names))))
    print(f"  {held:8s} | {balanced_accuracy_score(lab[ev],pae.argmax(1)):.3f} | {ece(pae,lab[ev]):.3f} | {au:.3f} | {acc70:.3f} | {mh:6.1f} | {cnt}")
    rows.append([balanced_accuracy_score(lab[ev],pae.argmax(1)),ece(pae,lab[ev]),mh])
rows=np.array(rows)
print(f"  MEAN: bal.acc {rows[:,0].mean():.3f} | ECE {rows[:,1].mean():.3f} | Maha {rows[:,2].mean():.1f}  (n folds={len(rows)})")

ARM B (organ) over ['Epithelial', 'Lymphocyte', 'Other']
held-out | bal.acc | ECE  | AURC | acc@70 | Maha | held-out counts
  Breast   | 0.809 | 0.076 | 0.083 | 0.909 |  444.3 | {'Epithelial': np.int64(5930), 'Lymphocyte': np.int64(6476), 'Other': np.int64(326)}
  Kidney   | 0.824 | 0.047 | 0.035 | 0.957 |  411.4 | {'Epithelial': np.int64(5785), 'Lymphocyte': np.int64(6202), 'Other': np.int64(380)}
  Lung     | 0.849 | 0.091 | 0.088 | 0.896 |  376.1 | {'Epithelial': np.int64(4467), 'Lymphocyte': np.int64(4677), 'Other': np.int64(417)}
  Prostate | 0.881 | 0.009 | 0.013 | 0.986 |  370.7 | {'Epithelial': np.int64(5539), 'Lymphocyte': np.int64(5926), 'Other': np.int64(561)}
  MEAN: bal.acc 0.841 | ECE 0.056 | Maha 400.6  (n folds=4)


### Appearance arm + gate — reload CONCH (these re-embed jittered crops)
Same pattern as your file-2 stain cell. Loads the cached crops from Notebook 1 and the **frozen**
scaler/probe/T/geometry above, so the only thing that changes is the stain appearance.

In [9]:
!GIT_TERMINAL_PROMPT=0 pip install -q "git+https://github.com/mahmoodlab/CONCH.git"
from conch.open_clip_custom import create_model_from_pretrained
from PIL import Image; from skimage.color import rgb2hed, hed2rgb
try:
    from kaggle_secrets import UserSecretsClient; tok=UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    import os; tok=os.environ.get("HF_TOKEN")
device="cuda" if torch.cuda.is_available() else "cpu"
cmodel,preprocess=create_model_from_pretrained("conch_ViT-B-16","hf_hub:MahmoodLab/conch",hf_auth_token=tok)
cmodel=cmodel.to(device).eval()

crops=np.load(glob.glob("/kaggle/input/**/monusac_crops.npy",recursive=True)[0] if glob.glob("/kaggle/input/**/monusac_crops.npy",recursive=True) else "/kaggle/working/monusac_crops.npy")
te_crops=crops[te]; assert len(te_crops)==te.sum()

@torch.inference_mode()
def embed(batch):
    t=torch.stack([preprocess(Image.fromarray(p)) for p in batch]).to(device)
    return cmodel.encode_image(t,proj_contrast=False,normalize=False).cpu().numpy()
def embed_all(cs): return np.concatenate([embed(cs[s:s+256]) for s in range(0,len(cs),256)])
def hed_jitter(img,sigma,rng):
    hed=rgb2hed(img/255.0); hed=hed*(1+rng.uniform(-sigma,sigma,3))+rng.uniform(-sigma,sigma,3)
    return np.clip(hed2rgb(hed)*255,0,255).astype(np.uint8)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incom

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


meta.yaml:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/802M [00:00<?, ?B/s]

In [10]:
# ---- ARM A: appearance (HED jitter) severity, multi-seed, using FROZEN reference ----
def stain_metrics(cs):
    Xs=scaler.transform(embed_all(cs)); paj=softmax(clf.decision_function(Xs)/T)
    ok=paj.argmax(1)==yte; _,_,a=risk_coverage(paj.max(1),ok)
    return (balanced_accuracy_score(yte,paj.argmax(1)),ece(paj,yte),a,float(maha_min(Xs,mus,P).mean()))
SIG=[0.05,0.10,0.20,0.35]; res={0.0:[stain_metrics(te_crops)]}
for sg in SIG: res[sg]=[stain_metrics(np.stack([hed_jitter(c,sg,np.random.RandomState(s)) for c in te_crops])) for s in range(5)]
print("ARM A -- expect bal.acc DOWN, ECE UP (silent), Maha UP (shouts)")
print("sigma |  bal.acc   |    ECE     |   AURC     |  mean Maha")
for sg in [0.0]+SIG:
    a=np.array(res[sg]); m,sd=a.mean(0),a.std(0)
    print(f" {sg:>4.2f} | {m[0]:.3f}±{sd[0]:.3f}| {m[1]:.3f}±{sd[1]:.3f}| {m[2]:.3f}±{sd[2]:.3f}| {m[3]:7.1f}±{sd[3]:5.1f}")

ARM A -- expect bal.acc DOWN, ECE UP (silent), Maha UP (shouts)
sigma |  bal.acc   |    ECE     |   AURC     |  mean Maha
 0.00 | 0.807±0.000| 0.026±0.000| 0.022±0.000|   354.3±  0.0
 0.05 | 0.679±0.087| 0.116±0.085| 0.140±0.103|   884.5±504.4
 0.10 | 0.472±0.163| 0.502±0.322| 0.599±0.402|  1828.2±1171.7
 0.20 | 0.382±0.125| 0.675±0.337| 0.743±0.360|  3522.4±1712.5
 0.35 | 0.321±0.085| 0.711±0.332| 0.788±0.354|  5315.1±2319.9


In [11]:
# ---- GATE: Mahalanobis-gated switch vs per-regime oracle (the journal contribution) ----
TAU=np.percentile(maha_min(scaler.transform(embed_all(te_crops)),mus,P),95)
def gate_regime(cs,label):
    Xs=scaler.transform(embed_all(cs)); paj=softmax(clf.decision_function(Xs)/T)
    conf=paj.max(1); mh=maha_min(Xs,mus,P); ok=paj.argmax(1)==yte
    _,_,a_conf=risk_coverage(conf,ok); _,_,a_geo=risk_coverage(-mh,ok)
    trust=np.where(mh>TAU,-mh,conf-conf.min()); _,_,a_sw=risk_coverage(trust,ok)
    orc=min(a_conf,a_geo)
    print(f"  {label:13s}: AURC conf={a_conf:.3f} geo={a_geo:.3f} switch={a_sw:.3f} oracle={orc:.3f} | recovery={orc/max(a_sw,1e-9):.2f}")
print("GATE -- recovery>=1.0 means the switch matches the per-regime oracle WITHOUT knowing the regime")
gate_regime(te_crops,"in-dist")
gate_regime(np.stack([hed_jitter(c,0.20,np.random.RandomState(0)) for c in te_crops]),"stain σ=0.20")

GATE -- recovery>=1.0 means the switch matches the per-regime oracle WITHOUT knowing the regime
  in-dist      : AURC conf=0.022 geo=0.049 switch=0.021 oracle=0.022 | recovery=1.06
  stain σ=0.20 : AURC conf=0.038 geo=0.160 switch=0.147 oracle=0.038 | recovery=0.26
